In [3]:
pip install textblob

   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------- ----------------------- 262.1/625.0 kB ? eta -:--:--
   ---------------------------------------- 625.0/625.0 kB 2.4 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob

In [ ]:
FILE = "commonwealth_bank_social_media_dataset.xlsx"

df = pd.read_excel(r'E:\DS\tezendra\Projects\Job Simulation of Common Wealth\Task 3\commonwealth_bank_social_media_dataset.xlsx')


def get_polarity(text):
    return TextBlob(str(text)).sentiment.polarity


def classify(polarity):
    if polarity > 0.15:
        return "Positive"
    elif polarity < -0.15:
        return "Negative"
    return "Neutral"


# The initial read loaded the workbook's introductory sheet.
# Reload the actual data sheet when post_text is unavailable.
if "post_text" not in df.columns:
    df = pd.read_excel(
        r"E:\DS\tezendra\Projects\Job Simulation of Common Wealth\Task 3\commonwealth_bank_social_media_dataset.xlsx",
        sheet_name="Social_Media_Posts"
    )

df["model_sentiment_score"] = df["post_text"].fillna("").apply(get_polarity)
df["model_sentiment_label"] = df["model_sentiment_score"].apply(classify)

df["labels_match"] = (
    df["sentiment_label"] == df["model_sentiment_label"]
)

agreement_rate = df["labels_match"].mean()

print("Model vs. curated label agreement")
print(f"Agreement rate: {agreement_rate:.1%}\n")

print("Curated sentiment distribution")
print(df["sentiment_label"].value_counts(), "\n")

print("Model sentiment distribution")
print(df["model_sentiment_label"].value_counts(), "\n")

by_topic = (
    df.groupby("topic")["sentiment_score"]
    .agg(["mean", "count"])
    .sort_values("mean")
    .rename(columns={
        "mean": "avg_sentiment",
        "count": "post_count"
    })
)

print("Average sentiment by topic")
print(by_topic, "\n")

by_type = (
    df.groupby("post_type")["sentiment_score"]
    .mean()
    .sort_values()
)

print("Average sentiment by post type")
print(by_type, "\n")

df["post_date"] = pd.to_datetime(df["post_datetime"]).dt.date

by_date = df.groupby("post_date")["sentiment_score"].mean()

print("Average sentiment by date")
print(by_date, "\n")

pain_points = df[
    (df["sentiment_label"] == "Negative") &
    (df["response_needed"] == "Yes")
][[
    "post_id",
    "topic",
    "customer_intent",
    "post_text"
]]

print("Negative posts needing a response")
print(pain_points.to_string(index=False), "\n")

df.to_csv("sentiment_analysis_output.csv", index=False)

print("Saved sentiment_analysis_output.csv")

KeyError: 'post_text'